In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load all tables
app_train = pd.read_csv('../data/raw/application_train.csv', encoding='cp1252')
app_test  = pd.read_csv('../data/raw/application_test.csv',  encoding='cp1252')
bureau    = pd.read_csv('../data/raw/bureau.csv',            encoding='cp1252')
bb        = pd.read_csv('../data/raw/bureau_balance.csv',    encoding='cp1252')
prev      = pd.read_csv('../data/raw/previous_application.csv', encoding='cp1252')
inst      = pd.read_csv('../data/raw/installments_payments.csv', encoding='cp1252')
cc        = pd.read_csv('../data/raw/credit_card_balance.csv',   encoding='cp1252')
pos       = pd.read_csv('../data/raw/POS_CASH_balance.csv',      encoding='cp1252')

print("Shapes:")
for name, df in zip(['app_train','bureau','bb','prev','inst','cc','pos'],
                    [app_train, bureau, bb, prev, inst, cc, pos]):
    print(f"  {name}: {df.shape}")

Shapes:
  app_train: (307511, 122)
  bureau: (1716428, 17)
  bb: (27299925, 3)
  prev: (1670214, 37)
  inst: (13605401, 8)
  cc: (3840312, 23)
  pos: (10001358, 8)


In [2]:
# bureau_balance содержит помесячные статусы по каждому кредиту в бюро
# Статусы: C=closed, X=unknown, 0-5 (просрочка в месяцах)

# Encode STATUS as numeric severity
status_map = {'C': 0, 'X': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bb['STATUS_NUM'] = bb['STATUS'].map(status_map)

bb_agg = bb.groupby('SK_ID_BUREAU').agg(
    bb_months_count   = ('MONTHS_BALANCE', 'count'),
    bb_max_dpd        = ('STATUS_NUM', 'max'),   # worst ever delinquency
    bb_mean_dpd       = ('STATUS_NUM', 'mean'),  # avg delinquency
    bb_dpd_months     = ('STATUS_NUM', lambda x: (x > 0).sum()),  # months with any DPD
).reset_index()

print(bb_agg.shape)
bb_agg.head()

(817395, 5)


,SK_ID_BUREAU,bb_months_count,bb_max_dpd,bb_mean_dpd,bb_dpd_months
0,5001709,97,0,0.0,0
1,5001710,83,0,0.0,0
2,5001711,4,0,0.0,0
3,5001712,19,0,0.0,0
4,5001713,22,0,0.0,0


In [3]:
# Join bureau with bureau_balance aggregates
bureau_full = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')

bureau_agg = bureau_full.groupby('SK_ID_CURR').agg(
    # Credit counts
    bureau_loan_count         = ('SK_ID_BUREAU', 'count'),
    bureau_active_count       = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    bureau_closed_count       = ('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),

    # Credit amounts
    bureau_debt_sum           = ('AMT_CREDIT_SUM_DEBT', 'sum'),
    bureau_credit_sum         = ('AMT_CREDIT_SUM', 'sum'),
    bureau_overdue_sum        = ('AMT_CREDIT_SUM_OVERDUE', 'sum'),

    # Delinquency history
    bureau_max_overdue        = ('AMT_CREDIT_MAX_OVERDUE', 'max'),
    bureau_days_credit_mean   = ('DAYS_CREDIT', 'mean'),
    bureau_days_enddate_max   = ('DAYS_CREDIT_ENDDATE', 'max'),

    # From bureau_balance
    bureau_bb_max_dpd         = ('bb_max_dpd', 'max'),
    bureau_bb_mean_dpd        = ('bb_mean_dpd', 'mean'),
    bureau_bb_dpd_months_sum  = ('bb_dpd_months', 'sum'),
).reset_index()

print(bureau_agg.shape)
bureau_agg.head()

(305811, 13)


,SK_ID_CURR,bureau_loan_count,bureau_active_count,bureau_closed_count,bureau_debt_sum,bureau_credit_sum,bureau_overdue_sum,bureau_max_overdue,bureau_days_credit_mean,bureau_days_enddate_max,bureau_bb_max_dpd,bureau_bb_mean_dpd,bureau_bb_dpd_months_sum
0,100001,7,3,4,596686.5,1453365.000,0.0,NaN,-735.000000,1778.0,1.0,0.007519,1.0
1,100002,8,2,6,245781.0,865055.565,0.0,5043.645,-874.000000,780.0,1.0,0.255682,27.0
2,100003,4,1,3,0.0,1017400.500,0.0,0.000,-1400.750000,1216.0,NaN,NaN,0.0
3,100004,2,0,2,0.0,189037.800,0.0,0.000,-867.000000,-382.0,NaN,NaN,0.0
4,100005,3,2,1,568408.5,657126.000,0.0,0.000,-190.666667,1324.0,0.0,0.000000,0.0


In [6]:
prev[['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']].describe()

,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION
count,997149.000000,997149.000000,997149.000000,997149.000000,997149.000000
mean,342209.855039,13826.269337,33767.774054,76582.403064,81992.343838
std,88916.115833,72444.869708,106857.034789,149647.415123,153303.516729
min,-2922.000000,-2892.000000,-2801.000000,-2889.000000,-2874.000000
25%,365243.000000,-1628.000000,-1242.000000,-1314.000000,-1270.000000
50%,365243.000000,-831.000000,-361.000000,-537.000000,-499.000000
75%,365243.000000,-411.000000,129.000000,-74.000000,-44.000000
max,365243.000000,365243.000000,365243.000000,365243.000000,365243.000000


In [ ]:
# Replace anomalous values
prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan, inplace=True)
prev['DAYS_FIRST_DUE'].replace(365243, np.nan, inplace=True)
prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan, inplace=True)
prev['DAYS_LAST_DUE'].replace(365243, np.nan, inplace=True)
prev['DAYS_TERMINATION'].replace(365243, np.nan, inplace=True)

# Credit utilization ratio for previous loans
prev['PREV_CREDIT_UTIL'] = prev['AMT_APPLICATION'] / (prev['AMT_CREDIT'] + 1)

prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_app_count             = ('SK_ID_PREV', 'count'),
    prev_approved_count        = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    prev_refused_count         = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    prev_amt_credit_mean       = ('AMT_CREDIT', 'mean'),
    prev_amt_annuity_mean      = ('AMT_ANNUITY', 'mean'),
    prev_amt_down_payment_mean = ('AMT_DOWN_PAYMENT', 'mean'),
    prev_credit_util_mean      = ('PREV_CREDIT_UTIL', 'mean'),
    prev_days_decision_min     = ('DAYS_DECISION', 'min'),
).reset_index()

# Approval rate
prev_agg['prev_approval_rate'] = (
    prev_agg['prev_approved_count'] / prev_agg['prev_app_count']
)

print(prev_agg.shape)
prev_agg.head()

(338857, 10)


,SK_ID_CURR,prev_app_count,prev_approved_count,prev_refused_count,prev_amt_credit_mean,prev_amt_annuity_mean,prev_amt_down_payment_mean,prev_credit_util_mean,prev_days_decision_min,prev_approval_rate
0,100001,1,1,0,23787.00,3951.000,2520.0,1.044035,-1740,1.0
1,100002,1,1,0,179055.00,9251.775,0.0,0.999994,-606,1.0
2,100003,3,3,0,484191.00,56553.990,3442.5,0.949323,-2341,1.0
3,100004,1,1,0,20106.00,5357.250,4860.0,1.207639,-815,1.0
4,100005,2,1,0,20076.75,4813.200,4464.0,0.555573,-757,0.5


In [8]:
# Payment behavior: did they pay on time, how much?
inst['DAYS_LATE'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['AMT_UNDERPAID'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']

inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_count                = ('SK_ID_PREV', 'count'),
    inst_days_late_mean       = ('DAYS_LATE', 'mean'),
    inst_days_late_max        = ('DAYS_LATE', 'max'),
    inst_late_payments_count  = ('DAYS_LATE', lambda x: (x > 0).sum()),
    inst_amt_underpaid_mean   = ('AMT_UNDERPAID', 'mean'),
    inst_amt_underpaid_sum    = ('AMT_UNDERPAID', 'sum'),
).reset_index()

# Late payment rate
inst_agg['inst_late_payment_rate'] = (
    inst_agg['inst_late_payments_count'] / inst_agg['inst_count']
)

print(inst_agg.shape)
inst_agg.head()

(339587, 8)


,SK_ID_CURR,inst_count,inst_days_late_mean,inst_days_late_max,inst_late_payments_count,inst_amt_underpaid_mean,inst_amt_underpaid_sum,inst_late_payment_rate
0,100001,7,-7.285714,11.0,1,0.0,0.0,0.142857
1,100002,19,-20.421053,-12.0,0,0.0,0.0,0.000000
2,100003,25,-7.160000,-1.0,0,0.0,0.0,0.000000
3,100004,3,-7.666667,-3.0,0,0.0,0.0,0.000000
4,100005,9,-23.555556,1.0,1,0.0,0.0,0.111111
